# N01 — fidelidade numérica e estabilidade decisória

Notebook público e reproduzível da investigação **EXPLORATORY**. Ele não contém resultados, identificadores de execução, dados individuais ou outputs persistidos. Para analisar artefatos privados locais, o pesquisador deve fornecer explicitamente um Run ID no ambiente.

## Objetivo e metodologia

Comparar a árvore sklearn float com a mesma árvore adaptada ao Concrete ML em `disable`, `simulate` e, quando autorizado, FHE. O target, os recortes temporais e as features são exploratórios e definidos em `config.yaml`. O notebook não treina, compila nem executa FHE automaticamente.

As comparações devem ser pareadas: as mesmas observações devem alimentar referência e saída quantizada. Métricas de populações distintas não devem ser combinadas.

## Definições matemáticas

Para `d(p, τ) = 1[p ≤ τ]`, margem `m_i = |p_i-τ|` e erro `e_i = |q_i-p_i|`, `m_i > e_i` é condição suficiente para estabilidade individual; discordância requer `m_i ≤ e_i`. Para vetores finitos, a integral exata da DDR por cutoff é o MAE. Com pesos positivos normalizados, a integral da DDR ponderada é o MAE ponderado. Essas identidades descrevem uma propriedade matemática, não definem uma política financeira.

In [ ]:
import json
import os
from pathlib import Path

UNIT = Path('experiments/04_numerical_fidelity/N01_freddie_mac_decision_tree')
RUN_ID = os.environ.get('N01_RUN_ID')
if not RUN_ID:
    raise RuntimeError('Defina N01_RUN_ID para selecionar explicitamente um run privado local.')
RUN_DIR = UNIT / 'results' / 'runs' / RUN_ID
ANALYSIS_DIR = UNIT / 'results' / 'analyses' / RUN_ID

def load_json(path):
    with Path(path).open(encoding='utf-8') as stream:
        return json.load(stream)

manifest = load_json(RUN_DIR / 'manifest.json')
assert manifest['run_id'] == RUN_ID
assert manifest['status'] == 'COMPLETED'
print('Manifesto privado carregado e selecionado explicitamente.')

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

def predictive_metrics(y_true, probability):
    return {
        'roc_auc': float(roc_auc_score(y_true, probability)),
        'average_precision': float(average_precision_score(y_true, probability)),
        'brier': float(brier_score_loss(y_true, probability)),
        'log_loss': float(log_loss(y_true, probability, labels=[0, 1])),
    }

def fidelity_metrics(reference, quantized):
    error = np.asarray(quantized, dtype=float) - np.asarray(reference, dtype=float)
    return {
        'mae': float(np.mean(np.abs(error))),
        'rmse': float(np.sqrt(np.mean(error ** 2))),
        'max_absolute_error': float(np.max(np.abs(error))),
        'mean_signed_error': float(np.mean(error)),
        'distinct_probabilities': int(np.unique(quantized).size),
    }

# Use estas funções somente com vetores pareados carregados de artefatos privados autorizados.

In [ ]:
def ddr_intervals(reference, quantized, weights=None):
    reference = np.asarray(reference, dtype=float)
    quantized = np.asarray(quantized, dtype=float)
    weights = np.ones(reference.size) if weights is None else np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    boundaries = np.unique(np.r_[0.0, 1.0, reference, quantized])
    rows, area = [], 0.0
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        cutoff = (left + right) / 2
        disagreement = (reference <= cutoff) != (quantized <= cutoff)
        ddr = float(np.dot(weights, disagreement))
        rows.append({'left': float(left), 'right': float(right), 'ddr': ddr})
        area += ddr * (right - left)
    mae = float(np.dot(weights, np.abs(quantized - reference)))
    return rows, {'area_under_ddr': float(area), 'mae': mae, 'identity_error': float(area - mae)}

# A convenção é aprovação quando p <= cutoff; use pesos apenas quando sua semântica estiver documentada.

In [ ]:
import matplotlib.pyplot as plt

def plot_ddr_step(intervals, ax=None, label=None):
    ax = ax or plt.subplots(figsize=(8, 3))[1]
    if intervals:
        edges = [intervals[0]['left']] + [row['right'] for row in intervals]
        ax.stairs([row['ddr'] for row in intervals], edges, label=label)
    ax.set(xlabel='cutoff τ', ylabel='DDR', title='Discordância decisória por cutoff')
    if label:
        ax.legend()
    return ax

def plot_precision_summary(bits, mae, ddr_max, distinct):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    axes[0].plot(bits, mae, marker='o'); axes[0].set(xlabel='n_bits', ylabel='MAE')
    axes[1].step(bits, ddr_max, where='mid'); axes[1].set(xlabel='n_bits', ylabel='DDR máxima')
    axes[2].step(bits, distinct, where='mid'); axes[2].set(xlabel='n_bits', ylabel='saídas distintas')
    fig.tight_layout()
    return fig

# Popule os argumentos com resumos calculados localmente; não persista resultados no notebook.

## Discussão, limitações e reprodução

Documente separadamente referência float, quantização em claro, simulação e FHE real. Preservação de ranking não implica fidelidade cardinal nem estabilidade para todos os cutoffs. Não generalize resultados de uma árvore, um período ou poucas observações FHE. Original UPB, se empregado como peso, é exposição contratual inicial e não EAD prospectiva.

Para reproduzir localmente, prepare o ambiente autorizado, execute o experimento pelo executor central somente com autorização explícita e defina `N01_RUN_ID=<run-id-local>` antes de abrir este notebook. Não publique dados, modelos, predições, chaves, circuitos, logs ou análises privadas.